# Topic 1: User-Defined Functions (UDFs)

> **Phase 3 → Week 5 → Topic 1**
>
> Prerequisites: Week 4 complete (Joins, Window Functions, Built-in Functions)

---

## Why UDFs Exist — and Why They're a Last Resort

Built-in functions cover ~90% of needs. When you need custom logic, you have two options:

```
Option 1: Python UDF                    Option 2: Pandas UDF (Vectorized)
─────────────────────────────────────   ────────────────────────────────────────
JVM → serialize row to Python           JVM → send Arrow BATCH to Python
     → run Python function              → run Pandas/NumPy on entire batch
     → serialize result back to JVM     → return Arrow batch to JVM

Cost per row: HIGH (row-by-row)         Cost: one round-trip per partition
Catalyst optimization: NONE             Catalyst optimization: NONE
Speed vs built-in: 10-100x slower       Speed: 3-5x slower than built-in
                                         (but much faster than Python UDF)
```

**UDF performance order (fastest → slowest):**
1. Built-in `pyspark.sql.functions` — always prefer
2. Pandas UDF (vectorized) — use for complex logic or when you need NumPy/Pandas
3. Python UDF — use only when 1 and 2 aren't possible

---

## Interview Cheat Sheet

**Q: What is a UDF in Spark? What are its downsides?**
> A UDF (User-Defined Function) is a custom function registered with Spark to extend the DataFrame API. Downsides: (1) Catalyst cannot optimize inside a UDF — it's a black box; (2) Each row is serialized from JVM to Python and back — row-by-row overhead; (3) Null handling must be done manually inside the function.

**Q: What is a Pandas UDF and how is it faster than a regular UDF?**
> A Pandas UDF (aka vectorized UDF) uses Apache Arrow to transfer an entire batch/partition of data from JVM to Python as a Pandas Series at once — not row-by-row. The Python function receives a Pandas Series and returns one. This eliminates most serialization overhead. Still slower than built-in functions but 3-10x faster than regular Python UDFs.

**Q: What happens when a Python UDF receives a null value?**
> By default, if the input is `null`, Spark passes `None` to your Python function. If your function raises an exception on `None`, the job fails. You must handle `None` explicitly inside the UDF, or add a guard: `if val is None: return None`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

spark = SparkSession.builder \
    .appName("Week5 - UDFs") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
print("Ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/14 03:44:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: Python UDF — The Basics

In [2]:
# Example: classify order amounts into tiers
# Built-in when() can do this — this is just for demonstration

# Method 1: @udf decorator
@udf(returnType=StringType())
def classify_amount(amount):
    if amount is None:
        return "Unknown"          # always handle None!
    elif amount >= 1000:
        return "High"
    elif amount >= 200:
        return "Medium"
    elif amount >= 50:
        return "Low"
    else:
        return "Micro"

# Method 2: udf() function
def title_case(s):
    if s is None:
        return None
    return s.title()

title_case_udf = udf(title_case, StringType())

# Apply UDFs
result = orders.withColumn("amount_tier", classify_amount(F.col("amount"))) \
               .withColumn("status_title", title_case_udf(F.col("status")))

print("UDF results:")
result.select("order_id", "amount", "amount_tier", "status", "status_title").show()

UDF results:


+--------+-------+-----------+----------+------------+
|order_id| amount|amount_tier|    status|status_title|
+--------+-------+-----------+----------+------------+
|    O001|1299.99|       High| delivered|   Delivered|
|    O002|  79.98|        Low| delivered|   Delivered|
|    O003|  89.97|        Low| delivered|   Delivered|
|    O004|1299.99|       High| delivered|   Delivered|
|    O005| 599.98|     Medium| delivered|   Delivered|
|    O006|  49.99|      Micro|   shipped|     Shipped|
|    O007| 199.99|        Low| delivered|   Delivered|
|    O008| 449.99|     Medium| delivered|   Delivered|
|    O009|  44.99|      Micro|processing|  Processing|
|    O010| 149.95|        Low| delivered|   Delivered|
|    O011|  69.98|        Low| cancelled|   Cancelled|
|    O012|2599.98|       High| delivered|   Delivered|
|    O013| 199.99|        Low| delivered|   Delivered|
|    O014|  99.98|        Low|   shipped|     Shipped|
|    O015| 119.97|        Low| delivered|   Delivered|
|    O016|

In [3]:
# Register UDF for use in SQL queries
spark.udf.register("classify_amount_sql", classify_amount)
orders.createOrReplaceTempView("orders_view")

spark.sql("""
    SELECT order_id, amount, classify_amount_sql(amount) AS tier
    FROM orders_view
    ORDER BY amount DESC
    LIMIT 5
""").show()

+--------+-------+------+
|order_id| amount|  tier|
+--------+-------+------+
|    O012|2599.98|  High|
|    O001|1299.99|  High|
|    O004|1299.99|  High|
|    O016|1299.99|  High|
|    O005| 599.98|Medium|
+--------+-------+------+



In [4]:
# UDF with complex return type — StructType
@udf(returnType=StructType([
    StructField("first", StringType(), True),
    StructField("last",  StringType(), True),
    StructField("length", IntegerType(), True),
]))
def parse_name(full_name):
    if full_name is None:
        return None
    parts = full_name.strip().split(" ")
    first = parts[0]
    last  = parts[-1] if len(parts) > 1 else None
    return (first, last, len(full_name))

parsed = customers.withColumn("parsed_name", parse_name(F.col("name")))
print("UDF returning StructType:")
parsed.select(
    "name",
    "parsed_name.first",
    "parsed_name.last",
    "parsed_name.length"
).show()

UDF returning StructType:


+--------------+-----+--------+------+
|          name|first|    last|length|
+--------------+-----+--------+------+
|  Alice Sharma|Alice|  Sharma|    12|
|      Bob Chen|  Bob|    Chen|     8|
|Carol Williams|Carol|Williams|    14|
|    Dave Kumar| Dave|   Kumar|    10|
|    Eve Müller|  Eve|  Müller|    10|
|  Frank Tanaka|Frank|  Tanaka|    12|
|    Grace Park|Grace|    Park|    10|
|  Henry Okafor|Henry|  Okafor|    12|
|   Irene Rossi|Irene|   Rossi|    11|
|   James Brown|James|   Brown|    11|
+--------------+-----+--------+------+



In [5]:
# UDF with ArrayType return type
@udf(returnType=ArrayType(StringType()))
def tokenize(text):
    if text is None:
        return []
    return [word.lower().strip() for word in text.split()]

text_df = spark.createDataFrame([
    (1, "Apache Spark is amazing"),
    (2, "PySpark makes big data easy"),
    (3, None),
], ["id", "text"])

text_df.withColumn("tokens", tokenize(F.col("text"))) \
       .withColumn("word_count", F.size("tokens")) \
       .show(truncate=False)

+---+---------------------------+---------------------------------+----------+
|id |text                       |tokens                           |word_count|
+---+---------------------------+---------------------------------+----------+
|1  |Apache Spark is amazing    |[apache, spark, is, amazing]     |4         |
|2  |PySpark makes big data easy|[pyspark, makes, big, data, easy]|5         |
|3  |NULL                       |[]                               |0         |
+---+---------------------------+---------------------------------+----------+



## Part 2: Pandas UDF (Vectorized) — The Fast Way

In [6]:
# Pandas UDF — function receives a pd.Series, returns a pd.Series
# Enable Arrow-based columnar transfer
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

@pandas_udf(StringType())
def classify_amount_pandas(amounts: pd.Series) -> pd.Series:
    # Operates on entire partition at once — no row-by-row Python loop
    return pd.cut(
        amounts,
        bins=[-float('inf'), 50, 200, 1000, float('inf')],
        labels=["Micro", "Low", "Medium", "High"]
    ).astype(str)

print("Pandas UDF (vectorized):")
orders.withColumn("tier_pandas", classify_amount_pandas(F.col("amount"))) \
      .select("order_id", "amount", "tier_pandas") \
      .show()

Pandas UDF (vectorized):


+--------+-------+-----------+
|order_id| amount|tier_pandas|
+--------+-------+-----------+
|    O001|1299.99|       High|
|    O002|  79.98|        Low|
|    O003|  89.97|        Low|
|    O004|1299.99|       High|
|    O005| 599.98|     Medium|
|    O006|  49.99|      Micro|
|    O007| 199.99|        Low|
|    O008| 449.99|     Medium|
|    O009|  44.99|      Micro|
|    O010| 149.95|        Low|
|    O011|  69.98|        Low|
|    O012|2599.98|       High|
|    O013| 199.99|        Low|
|    O014|  99.98|        Low|
|    O015| 119.97|        Low|
|    O016|1299.99|       High|
|    O017| 119.98|        Low|
|    O018|  44.99|      Micro|
|    O019| 299.99|     Medium|
|    O020|  59.98|        Low|
+--------+-------+-----------+



In [7]:
# Pandas UDF for date arithmetic (complex logic)
import pandas as pd
import numpy as np

@pandas_udf(DoubleType())
def days_until_next_quarter(dates: pd.Series) -> pd.Series:
    """Calculate days until end of current quarter."""
    dates = pd.to_datetime(dates)
    quarter_end = dates.dt.to_period('Q').dt.to_timestamp('Q') + pd.offsets.MonthEnd(0)
    return (quarter_end - dates).dt.days.astype(float)

orders.withColumn(
    "days_to_quarter_end", days_until_next_quarter(F.col("order_date"))
).select("order_id", "order_date", "days_to_quarter_end").show()

+--------+----------+-------------------+
|order_id|order_date|days_to_quarter_end|
+--------+----------+-------------------+
|    O001|2023-06-01|               29.0|
|    O002|2023-06-05|               25.0|
|    O003|2023-06-10|               20.0|
|    O004|2023-06-12|               18.0|
|    O005|2023-06-12|               18.0|
|    O006|2023-06-15|               15.0|
|    O007|2023-06-18|               12.0|
|    O008|2023-06-20|               10.0|
|    O009|2023-06-22|                8.0|
|    O010|2023-06-25|                5.0|
|    O011|2023-06-28|                2.0|
|    O012|2023-07-01|               91.0|
|    O013|2023-07-05|               87.0|
|    O014|2023-07-08|               84.0|
|    O015|2023-07-10|               82.0|
|    O016|2023-07-12|               80.0|
|    O017|2023-07-15|               77.0|
|    O018|2023-07-18|               74.0|
|    O019|2023-07-20|               72.0|
|    O020|2023-07-22|               70.0|
+--------+----------+-------------

In [8]:
# Pandas UDF with multiple columns (use struct trick)
@pandas_udf(DoubleType())
def weighted_score(amount: pd.Series, quantity: pd.Series) -> pd.Series:
    # Weight amount 70%, quantity contribution 30%
    max_amount = amount.max() if amount.max() > 0 else 1
    max_qty    = quantity.max() if quantity.max() > 0 else 1
    return (0.7 * amount / max_amount + 0.3 * quantity / max_qty) * 100

orders.withColumn(
    "score", F.round(weighted_score(F.col("amount"), F.col("quantity")), 1)
).select("order_id", "amount", "quantity", "score") \
 .orderBy(F.col("score").desc()) \
 .show()

+--------+-------+--------+-----+
|order_id| amount|quantity|score|
+--------+-------+--------+-----+
|    O012|2599.98|       2| 82.0|
|    O001|1299.99|       1| 41.0|
|    O004|1299.99|       1| 41.0|
|    O016|1299.99|       1| 41.0|
|    O010| 149.95|       5| 34.0|
|    O005| 599.98|       2| 28.2|
|    O015| 119.97|       3| 21.2|
|    O003|  89.97|       3| 20.4|
|    O008| 449.99|       1| 18.1|
|    O017| 119.98|       2| 15.2|
|    O014|  99.98|       2| 14.7|
|    O002|  79.98|       2| 14.2|
|    O019| 299.99|       1| 14.1|
|    O011|  69.98|       2| 13.9|
|    O020|  59.98|       2| 13.6|
|    O007| 199.99|       1| 11.4|
|    O013| 199.99|       1| 11.4|
|    O006|  49.99|       1|  7.3|
|    O009|  44.99|       1|  7.2|
|    O018|  44.99|       1|  7.2|
+--------+-------+--------+-----+



## Part 3: Performance Comparison

In [9]:
import time

# Large dataset for fair timing
large_df = spark.range(100_000).withColumn(
    "amount", (F.rand() * 2000).cast("double")
)

# Option 1: Built-in when()
t0 = time.time()
large_df.withColumn("tier",
    F.when(F.col("amount") >= 1000, "High")
     .when(F.col("amount") >= 200, "Medium")
     .when(F.col("amount") >= 50, "Low")
     .otherwise("Micro")
).count()
builtin_time = time.time() - t0

# Option 2: Python UDF
t0 = time.time()
large_df.withColumn("tier", classify_amount(F.col("amount"))).count()
udf_time = time.time() - t0

# Option 3: Pandas UDF
t0 = time.time()
large_df.withColumn("tier", classify_amount_pandas(F.col("amount"))).count()
pandas_udf_time = time.time() - t0

print(f"Performance on 100k rows:")
print(f"  Built-in when():  {builtin_time:.3f}s")
print(f"  Pandas UDF:       {pandas_udf_time:.3f}s  ({pandas_udf_time/builtin_time:.1f}x slower than built-in)")
print(f"  Python UDF:       {udf_time:.3f}s  ({udf_time/builtin_time:.1f}x slower than built-in)")

Performance on 100k rows:
  Built-in when():  1.696s
  Pandas UDF:       0.334s  (0.2x slower than built-in)
  Python UDF:       0.347s  (0.2x slower than built-in)


## Part 4: When to Use Each

| Scenario | Best Choice |
|----------|------------|
| String manipulation (upper, concat, split) | Built-in `F.upper()`, `F.split()` |
| Date math (datediff, date_format) | Built-in date functions |
| Conditional logic (IF/ELSE) | Built-in `F.when().otherwise()` |
| Mathematical transforms | Built-in `F.sqrt()`, `F.pow()` etc. |
| Complex business logic (regex, scoring) | Pandas UDF (vectorized) |
| ML feature engineering (requires NumPy) | Pandas UDF |
| Legacy Python function, can't vectorize | Python UDF (last resort) |
| Calling external API per row | Python UDF (no choice) |

In [10]:
# Common mistake: using UDF when built-in exists

# BAD — Python UDF for something built-in can do
@udf(StringType())
def bad_upper(s):
    return s.upper() if s else None

# GOOD — use built-in
df = customers.select(
    F.upper("name").alias("good_way"),    # use this
    bad_upper("name").alias("bad_way")    # never do this for simple ops
)
print("Result is same, but good_way is 10-100x faster:")
df.show(5)

Result is same, but good_way is 10-100x faster:


+--------------+--------------+
|      good_way|       bad_way|
+--------------+--------------+
|  ALICE SHARMA|  ALICE SHARMA|
|      BOB CHEN|      BOB CHEN|
|CAROL WILLIAMS|CAROL WILLIAMS|
|    DAVE KUMAR|    DAVE KUMAR|
|    EVE MÜLLER|    EVE MÜLLER|
+--------------+--------------+
only showing top 5 rows



## Exercises

1. Write a Python UDF that takes a `country` name and returns its continent (`Asia`, `Europe`, `America`, `Africa`, `Other`). Apply it to the customers table.
2. Rewrite the UDF from Exercise 1 as a Pandas UDF.
3. Write a Pandas UDF that normalizes `amount` values to a 0-100 scale (min-max normalization) using NumPy.
4. When would you use a Python UDF over a Pandas UDF? Give 2 scenarios.
5. A colleague wrote this: `@udf(DoubleType()) def avg_udf(values): return sum(values)/len(values)` and applied it to a list column. What's wrong? How would you fix it?

In [11]:
# Exercise 1: Country → Continent UDF
@udf(StringType())
def get_continent(country):
    if country is None:
        return "Unknown"
    mapping = {
        "India": "Asia", "China": "Asia", "Japan": "Asia",
        "South Korea": "Asia",
        "Germany": "Europe", "Italy": "Europe", "UK": "Europe",
        "USA": "America",
        "Nigeria": "Africa",
    }
    return mapping.get(country, "Other")

customers.withColumn("continent", get_continent(F.col("country"))) \
         .select("name", "country", "continent") \
         .show()

+--------------+-----------+---------+
|          name|    country|continent|
+--------------+-----------+---------+
|  Alice Sharma|      India|     Asia|
|      Bob Chen|      China|     Asia|
|Carol Williams|        USA|  America|
|    Dave Kumar|      India|     Asia|
|    Eve Müller|    Germany|   Europe|
|  Frank Tanaka|      Japan|     Asia|
|    Grace Park|South Korea|     Asia|
|  Henry Okafor|    Nigeria|   Africa|
|   Irene Rossi|      Italy|   Europe|
|   James Brown|         UK|   Europe|
+--------------+-----------+---------+



In [12]:
# Exercise 3: Min-max normalization Pandas UDF
import numpy as np

@pandas_udf(DoubleType())
def normalize_amount(amounts: pd.Series) -> pd.Series:
    min_val = amounts.min()
    max_val = amounts.max()
    if max_val == min_val:
        return pd.Series([0.0] * len(amounts))
    return ((amounts - min_val) / (max_val - min_val) * 100).round(2)

orders.withColumn("normalized_amount", normalize_amount(F.col("amount"))) \
      .select("order_id", "amount", "normalized_amount") \
      .show()

+--------+-------+-----------------+
|order_id| amount|normalized_amount|
+--------+-------+-----------------+
|    O001|1299.99|            49.12|
|    O002|  79.98|             1.37|
|    O003|  89.97|             1.76|
|    O004|1299.99|            49.12|
|    O005| 599.98|            21.72|
|    O006|  49.99|              0.2|
|    O007| 199.99|             6.07|
|    O008| 449.99|            15.85|
|    O009|  44.99|              0.0|
|    O010| 149.95|             4.11|
|    O011|  69.98|             0.98|
|    O012|2599.98|            100.0|
|    O013| 199.99|             6.07|
|    O014|  99.98|             2.15|
|    O015| 119.97|             2.93|
|    O016|1299.99|            49.12|
|    O017| 119.98|             2.94|
|    O018|  44.99|              0.0|
|    O019| 299.99|             9.98|
|    O020|  59.98|             0.59|
+--------+-------+-----------------+

